# TP 2 — Partie 4 : comprendre les avis avec le traitement du texte

**Question métier :** le service client reçoit des milliers de commentaires. Peut-on repérer **automatiquement** les clients mécontents à partir de ce qu'ils écrivent, pour les traiter en priorité ?

C'est une **classification de texte** : la cible est `est_negatif` (note 1 ou 2), l'entrée est le commentaire normalisé en partie 2.

La méthode, classique et efficace :
1. **TF-IDF** transforme chaque commentaire en vecteur de nombres : un poids par mot, d'autant plus fort que le mot est fréquent dans ce commentaire et rare dans les autres ;
2. une **régression logistique** apprend quels mots annoncent un avis négatif.

> **⚠️ Un seul notebook actif à la fois.** Arrêtez le noyau des autres notebooks avant de commencer.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pyspark.sql import SparkSession
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline

spark = SparkSession.builder.appName("tp2-nlp").getOrCreate()


def a_completer(quoi):
    """Signale clairement une partie du TP qui n'a pas encore été écrite."""
    raise NotImplementedError(f"⚠️ Cellule à compléter : {quoi}")


# Les avis commentés : un même avis peut porter sur plusieurs commandes, on le garde une fois
avis = (spark.table("lakehouse.silver.avis")
        .filter("nb_mots > 0")
        .select("review_id", "review_creation_date", "review_score", "est_negatif",
                "commentaire", "commentaire_normalise")
        .toPandas()
        .drop_duplicates("review_id"))
print(len(avis), "avis commentés |", f"{100 * avis.est_negatif.mean():.1f} % de négatifs")
avis.sample(5, random_state=1)[["review_score", "commentaire"]]

## Étape 1 — Les mots les plus fréquents

Les **mots vides** (*stop words*) sont les mots très fréquents qui ne portent pas de sens : articles, prépositions (« de », « o », « que »…). On les retire avant de compter. La liste est fournie, sans accents, comme les commentaires normalisés.

**Attention :** le mot **« nao »** (non) n'est volontairement **pas** dans la liste.

In [ ]:
MOTS_VIDES = """a ao aos as ate com como da das de dela delas dele deles depois do dos e ela elas ele
eles em entre era essa essas esse esses esta estao estas este estes eu foi foram ha isso isto ja lhe
lhes mais mas me mesmo meu meus minha minhas muito na nas no nos num numa o os ou para pela pelas
pelo pelos por qual quando que quem se sem ser seu seus so sua suas tambem te tem tenho ter teu tua
um uma voce voces vou""".split()


def mots_frequents(textes, n=12):
    cv = CountVectorizer(stop_words=MOTS_VIDES)
    totaux = np.asarray(cv.fit_transform(textes).sum(axis=0)).ravel()
    return list(cv.get_feature_names_out()[np.argsort(totaux)[::-1][:n]])


print("avis négatifs :", mots_frequents(avis.loc[avis.est_negatif, "commentaire_normalise"]))
print("avis positifs :", mots_frequents(avis.loc[~avis.est_negatif, "commentaire_normalise"]))

> **❓ Question 1.** Quels mots distinguent les deux listes ? « nao » arrive en tête des avis négatifs, mais il figure aussi dans les avis positifs : dans quelles expressions, selon vous ? (Petit lexique : *recebi* = j'ai reçu, *chegou* = est arrivé, *prazo* = délai, *entrega* = livraison, *recomendo* = je recommande, *otimo* = excellent.)

## Étape 2 — Découper dans le temps

Comme au TP 1, le modèle apprend sur le passé et il est évalué sur des avis plus récents : apprentissage avant le 1er avril 2018, test ensuite.

In [ ]:
coupure = pd.Timestamp("2018-04-01")
train = avis[avis.review_creation_date < coupure]
test = avis[avis.review_creation_date >= coupure]
X_train, y_train = train.commentaire_normalise, train.est_negatif
X_test, y_test = test.commentaire_normalise, test.est_negatif
print(f"apprentissage : {len(train)} avis | test : {len(test)} avis")
print(f"négatifs : {100 * y_train.mean():.1f} % (apprentissage), {100 * y_test.mean():.1f} % (test)")

## Étape 3 — Entraîner le modèle

Complétez la fonction `entrainer(ngrammes)`. Elle construit et entraîne un `Pipeline` scikit-learn en deux étapes :
1. `"tfidf"` : `TfidfVectorizer(stop_words=MOTS_VIDES, ngram_range=ngrammes, min_df=5, sublinear_tf=True)`. `ngram_range=(1, 2)` garde les mots seuls **et** les groupes de deux mots (« nao recomendo ») ; `min_df=5` ignore les mots vus dans moins de 5 avis ;
2. `"modele"` : `LogisticRegression(class_weight="balanced", max_iter=1000)` (`balanced` compense le fait qu'il y a moins d'avis négatifs).

Elle affiche ensuite l'AUC ROC et le rapport de classification sur le jeu de test.

In [ ]:
def entrainer(ngrammes):
    pipe = a_completer("pipeline TF-IDF + régression logistique, puis entraînement")  # À COMPLÉTER
    proba = pipe.predict_proba(X_test)[:, 1]
    print(f"===== n-grammes {ngrammes} — {len(pipe['tfidf'].vocabulary_)} termes — "
          f"AUC ROC : {roc_auc_score(y_test, proba):.3f}")
    print(classification_report(y_test, proba >= 0.5, target_names=["positif", "négatif"]))
    return pipe


mots_seuls = entrainer((1, 1))
modele = entrainer((1, 2))

> **❓ Question 2.** L'AUC dépasse 0,95, alors qu'elle plafonnait vers 0,7 pour les retards au TP 1. Pourquoi la tâche est-elle plus facile ici ? Les groupes de deux mots apportent-ils beaucoup ? Pour le service client, faut-il plutôt regarder le *rappel* ou la *précision* de la classe « négatif » ?

## Étape 4 — Ouvrir la boîte : les mots qui pèsent

Une régression logistique est **interprétable** : chaque terme a un coefficient. Positif, il pousse vers « négatif » ; négatif, il pousse vers « positif ».

In [ ]:
termes = modele["tfidf"].get_feature_names_out()
coefs = modele["modele"].coef_[0]
ordre = np.argsort(coefs)
print("annoncent un avis NÉGATIF :", list(termes[ordre[::-1][:15]]))
print("annoncent un avis POSITIF :", list(termes[ordre[:15]]))

## Étape 5 — Tester des phrases, et un piège

Testez le modèle sur vos propres phrases (en portugais, normalisées comme en partie 2 : minuscules, sans accents). Puis entraînez un modèle où « nao » **est** un mot vide, et comparez.

In [ ]:
phrases = ["produto chegou antes do prazo recomendo", "nao recebi o produto", "nao recomendo",
           "otimo", "produto muito bom mas a entrega atrasou"]
print(pd.DataFrame({"phrase": phrases, "proba_negatif": modele.predict_proba(phrases)[:, 1].round(2)}))

# À COMPLÉTER : pipeline identique, avec stop_words=MOTS_VIDES + ["nao"] ; afficher sa proba pour "nao recomendo"

> **❓ Question 3.** Que devient « nao recomendo » quand « nao » est un mot vide ? Qu'en concluez-vous sur les listes de mots vides toutes faites ? Que pensez-vous de la dernière phrase (« produit très bien mais la livraison a eu du retard ») ?

## Étape 6 — Les erreurs du modèle

Regardez quelques avis du jeu de test que le modèle classe avec assurance du mauvais côté.

In [ ]:
test = test.assign(proba_negatif=modele.predict_proba(X_test)[:, 1])
print("Notés négatifs, jugés positifs :")
print(test[test.est_negatif].nsmallest(5, "proba_negatif")[["review_score", "commentaire"]].to_string())
print("\nNotés positifs, jugés négatifs :")
print(test[~test.est_negatif].nlargest(5, "proba_negatif")[["review_score", "commentaire"]].to_string())

> **❓ Question 4.** Classez ces erreurs : le modèle se trompe-t-il, ou est-ce la note qui ne correspond pas au texte ? Donnez une limite de la méthode TF-IDF.

## Étape 7 — Écrire les prédictions en Gold et sauvegarder le modèle

Le modèle note **tous** les avis commentés. Les prédictions sont écrites dans `gold.avis_sentiment` (le tableau de bord les affichera), et le modèle est enregistré dans un fichier, que le tableau de bord rechargera pour noter un nouveau commentaire.

In [ ]:
avis = avis.assign(proba_negatif=modele.predict_proba(avis.commentaire_normalise)[:, 1].round(4))
resultat = avis[["review_id", "review_score", "est_negatif", "commentaire", "proba_negatif"]]

spark.createDataFrame(resultat).writeTo("lakehouse.gold.avis_sentiment").using("iceberg").createOrReplace()
joblib.dump(modele, "modele_avis.joblib")
print("gold.avis_sentiment :", spark.table("lakehouse.gold.avis_sentiment").count(), "avis | modèle : modele_avis.joblib")